# 03 · Train all models on Google Colab: Solar-Energy

Dinesh's notebook for **Solar** (Chandana's Electricity version: `03_train_on_colab.ipynb`).

Open in Colab:
`https://colab.research.google.com/github/Chandana18G/lstnet-reproduction/blob/main/notebooks/03_train_on_colab_solar.ipynb`

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells from top to bottom. Nothing to change: the dataset is fixed to Solar.

Everything is saved to **Google Drive** (`MyDrive/lstnet-checkpoints`) after every epoch.
If Colab stops, run all cells again: finished runs are skipped and an unfinished run continues
where it stopped. You can also split the work over several days.

## 1 · GPU check

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

## 2 · Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/lstnet-checkpoints"
import os
os.makedirs(CKPT, exist_ok=True)

## 3 · Code and data

Solar settings as in the authors' `solar.sh`: `--hid-skip 10`; horizon 24 as in the paper's
table (the script itself runs horizon 12, see `CLAUDE.md`).

In [ ]:
DATA = "solar"
EXTRA = "--hid-skip 10"         # authors' solar.sh; Electricity uses the default 5
%cd /content
!git clone -q https://github.com/Chandana18G/lstnet-reproduction.git || (cd lstnet-reproduction && git pull -q)
%cd /content/lstnet-reproduction
!bash scripts/download_data.sh {DATA}

## 4 · First: one LSTNet run (seed 0)

Run this first to see how long one run takes and whether we get close to the paper
(Solar: test RSE 0.4643, CORR 0.8870; our AR baseline: 0.5247). The first printed line must be
`run solar_h24_lstnet_skip24_seed0 on cuda`. Look at the time per epoch: 100 epochs × that time
= one run.

In [ ]:
import shutil, os
def run(model, seed):
    name = f"{DATA}_h24_{model}_skip24_seed{seed}"
    if os.path.exists(f"{CKPT}/{name}.json"):
        print("already done:", name)
        return
    get_ipython().system(f"python src/train.py --data {DATA} --horizon 24 --model {model} --seed {seed} {EXTRA} --checkpoint-dir {CKPT}")
    shutil.copy(f"results/runs/{name}.json", CKPT)      # keep the result on Drive

run("lstnet", 0)

## 5 · All models, seeds 0 to 9

Order: seed by seed, so that after any stop you have complete pairs (same seed for every
model). `lstnet` and `lstnet_revin` are the main comparison; `dlinear` is the simple linear
model; `lstnet_noar` is the ablation without AR.

In [ ]:
for seed in range(10):
    for model in ["lstnet", "lstnet_revin", "dlinear", "lstnet_noar"]:
        run(model, seed)

## 5b · Improvement (Dinesh): a one-day skip

One day = 144 steps, and the daily pattern is the strongest one in Solar (autocorrelation 0.85 at
144 steps, only 0.32 at our horizon of 24 steps; see `01_explore_solar.ipynb`). The paper's
skip-GRU uses p = 24 (4 hours), so it never compares a moment with the same moment yesterday.

`lstnet_skip144`: the same LSTNet, but `--skip 144`. A skip sequence has `(q - 6) // 144` steps,
so with the normal window q = 168 it would have only 1 step. We use q = 438 (3 days + 6) to get
3 steps per sequence: the same time today, yesterday and the day before.

Same 10 seeds, same loss, epochs and selection rule as the baseline. The models above are already
on Drive, so they are skipped. Then run cell 6 and the error analysis again.

In [ ]:
def run_day_skip(seed):
    """LSTNet with a one-day skip (144 steps) and a 3-day window (438 steps)."""
    name = f"{DATA}_h24_lstnet_skip144_seed{seed}"
    if os.path.exists(f"{CKPT}/{name}.json"):
        print("already done:", name)
        return
    get_ipython().system(f"python src/train.py --data {DATA} --horizon 24 --model lstnet --window 438 --skip 144 --seed {seed} {EXTRA} --checkpoint-dir {CKPT}")
    shutil.copy(f"results/runs/{name}.json", CKPT)      # keep the result on Drive

for seed in range(10):
    run_day_skip(seed)

## 6 · AR baseline and statistics

The AR baseline (needed for one comparison) runs on the CPU and takes a while for Solar
(L up to 512), so its predictions are kept on Drive and reused next time.
Then the statistics: mean ± std over seeds, paired tests over seeds, Diebold-Mariano.

In [ ]:
!git pull -q
!mkdir -p results/runs results/predictions && cp {CKPT}/{DATA}_*.json results/runs/
AR = f"{DATA}_h24_ar.npy"
if os.path.exists(f"{CKPT}/{AR}"):                    # computed in an earlier session
    shutil.copy(f"{CKPT}/{AR}", f"results/predictions/{AR}")
else:
    get_ipython().system(f"python src/baselines.py --data {DATA} --horizon 24")
    shutil.copy(f"results/predictions/{AR}", CKPT)
!python src/stats.py --data {DATA} --horizon 24 --pred-dir {CKPT}
shutil.copy(f"results/stats_{DATA}_h24.md", CKPT)

## Error analysis (report section 6)

Where the test errors are, from the predictions on Drive. The table and the figure are also
saved to Drive.

In [ ]:
!git pull -q
!python src/error_analysis.py --data {DATA} --horizon 24 --pred-dir {CKPT}
shutil.copy(f"results/error_analysis_{DATA}_h24.md", CKPT)
shutil.copy(f"results/figures/error_by_time_{DATA}.png", CKPT)

## 7 · Bring the results home

From `MyDrive/lstnet-checkpoints` download into your local repo:
- all `solar_*.json` files → `results/runs/`
- `stats_solar_h24.md` and `error_analysis_solar_h24.md` → `results/`
- `error_by_time_solar.png` → `results/figures/`

then commit. The `_test_predictions.npy`, `.pt` and `_ar.npy` files stay on Drive (too big for git).